In [1]:
%pip install langchain langchain-community langchainhub langchain-chroma langchain-openai langchain-text-splitters langchain-classic beautifulsoup4 yfinance gradio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 1

In [1]:
import os
import pprint
import getpass
import bs4
from bs4 import BeautifulSoup
from urllib.request import Request, urlopen
from google.colab import userdata

In [2]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import AIMessage, HumanMessage
from langchain_community.document_loaders import WebBaseLoader
from langchain_classic.chains import create_retrieval_chain, create_history_aware_retriever
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

import gradio as gr

In [4]:
# ---------------------------------------------------------
# 1. SETUP & DATA LOADING
# ---------------------------------------------------------
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

def get_sitemap(url):
    req = Request(
        url=url,
        headers={"User-Agent": "Mozilla/5.0"}
    )
    response = urlopen(req)
    xml = BeautifulSoup(
        response,
        "lxml-xml",
        from_encoding=response.info().get_param("charset")
    )
    return xml

def get_urls(xml, name=None, data=None, verbose=False):
    urls = []
    for url in xml.find_all("url"):
        if xml.find("loc"):
            loc = url.findNext("loc").text
            urls.append(loc)
    return urls

url = "https://zerodha.com/varsity/chapter-sitemap2.xml"
xml = get_sitemap(url)
urls = get_urls(xml, verbose=False)

docs = []
# Loading just the first 10 for speed; adjust as needed
for i, url in enumerate(urls[:10]):
    loader = WebBaseLoader(url)
    docs.extend(loader.load())
    if i % 10 == 0:
        print("Loaded document index:", i)

/tmp/ipython-input-977/3091849184.py:23: DeprecationWarning: Call to deprecated method findNext. (Replaced by find_next) -- Deprecated since version 4.0.0.
  loc = url.findNext("loc").text


Loaded document index: 0


In [5]:
# ---------------------------------------------------------
# 2. VECTORSTORE & RETRIEVER
# ---------------------------------------------------------
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)

vectorstore = Chroma.from_documents(documents=splits, embedding=OpenAIEmbeddings())
retriever = vectorstore.as_retriever()

print(f"Len docs: {len(docs)}, Len splits: {len(splits)}")

Len docs: 10, Len splits: 1175


In [6]:
# ---------------------------------------------------------
# 3. RAG CHAIN SETUP
# ---------------------------------------------------------
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0) # Updated to newer, cheaper model

system_prompt = (
    "You are a financial assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "If the question is not clear ask follow up questions"
    "\n\n"
    "{context}"
)

# History Aware Retriever
contextualize_q_system_prompt = (
    "Given a chat history and the latest user question "
    "which might reference context in the chat history, "
    "formulate a standalone question which can be understood "
    "without the chat history. Do NOT answer the question, "
    "just reformulate it if needed and otherwise return it as is."
)

contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_q_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

history_aware_retriever = create_history_aware_retriever(llm, retriever, contextualize_q_prompt)

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)
rag_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)

In [7]:
import gradio as gr
from langchain_core.messages import AIMessage, HumanMessage

def predict(message, history):
    """
    message: str
    history: list of dicts in OpenAI-style format when type='messages'
             e.g. [{"role":"user","content":"hi"}, {"role":"assistant","content":"hello"}]
    """
    history_for_llm = []
    for item in history:
        role = item.get("role")
        content = item.get("content", "")
        if role == "user":
            history_for_llm.append(HumanMessage(content=content))
        elif role == "assistant":
            history_for_llm.append(AIMessage(content=content))

    result = rag_chain.invoke({"input": message, "chat_history": history_for_llm})
    return result["answer"]

with gr.Blocks(theme="soft") as demo:
    gr.Markdown("# DocumentQABot")

    chatbot = gr.Chatbot(height=400, type="messages", allow_tags=False)
    msg = gr.Textbox(
        placeholder="Hi! I am your virtual assistant, how can I help you today?",
        container=False,
        scale=7,
    )

    with gr.Row():
        undo = gr.Button("Delete Previous")
        clear = gr.Button("Clear")

    chat = gr.ChatInterface(
        fn=predict,
        chatbot=chatbot,
        textbox=msg,
        examples=["What is index fund?", "Where to buy stocks?"],
        title=None,  # already using Markdown title
    )

    # Clear chat
    clear.click(lambda: [], None, chatbot)

    # Undo last turn (removes last 2 messages if present: user+assistant)
    def undo_last(history):
        if not history:
            return []
        # In messages format, history is a list of dicts, typically alternating user/assistant
        return history[:-2] if len(history) >= 2 else []

    undo.click(undo_last, chatbot, chatbot)

demo.launch(share=True, debug=False)

/tmp/ipython-input-977/2965842003.py:22: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme="soft") as demo:
/tmp/ipython-input-977/2965842003.py:25: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(height=400, type="messages", allow_tags=False)
/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:330: UserWarning: The gr.ChatInterface was not provided with a type, so the type of the gr.Chatbot, 'messages', will be used.
  warnings.warn(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://24a396d774b5464ef0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [8]:
import gradio as gr
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage
from langchain_community.tools.yahoo_finance_news import YahooFinanceNewsTool

tools = [YahooFinanceNewsTool()]
agent_test_prompt = "What is the latest news about Indian stock market like infosys"

# LangGraph ReAct agent
agent = create_react_agent(llm, tools)

print("\n--- Running Agent (LangGraph) ---")
result = agent.invoke({"messages": [HumanMessage(content=agent_test_prompt)]})

# The last assistant message content:
print(result["messages"][-1].content)

/tmp/ipython-input-977/108603975.py:10: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools)



--- Running Agent (LangGraph) ---
Here are the latest news updates regarding Infosys Limited (INFY):

1. **Collaboration with Anthropic**: Infosys has announced a collaboration with Anthropic to build AI agents for various industries. This partnership will integrate Infosys Topz, a suite of services and solutions based on generative AI, with Anthropic’s Claude Code and other technologies.

2. **Expansion of AI and Enterprise Partnerships**: Infosys is recognized as one of the best emerging market stocks to buy. The company has entered into a strategic partnership with Anthropic PBC to develop advanced artificial intelligence solutions. This collaboration aims to combine Anthropic's Claude Models with Infosys Topaz AI products to help businesses automate complex workflows and enhance efficiency.

These developments highlight Infosys's commitment to leveraging AI technology to drive innovation and improve business processes.
